# exp_001 — Context measurement

This notebook loads recorded trial JSONL, regenerates the processed summary, audits coverage, and plots measured position/context/system curves. It never invents missing cells or treats fixture smoke output as a Qwen finding.

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'src').is_dir() and (candidate / 'experiments').is_dir()
)
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'experiments/exp_001-context_measurement'))

from analyze import regenerate
from llm_lab.analysis import (
    aggregate_jsonl,
    effective_context_by_task,
    effective_context_by_task_and_position,
    position_gap_rows,
    position_curve_rows,
)

PHASE = os.environ.get('EXP001_PHASE', 'main')
ALLOW_FIXTURE = os.environ.get('EXP001_ALLOW_FIXTURE') == '1'
RESULTS_DIR = ROOT / 'experiments/exp_001-context_measurement/results'
RAW_PATH = RESULTS_DIR / 'raw' / f'{PHASE}-trials.jsonl'
SUMMARY_PATH = RESULTS_DIR / 'processed/summary.csv'
MANIFEST_PATH = RESULTS_DIR / 'manifests' / f'{PHASE}.json'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
regeneration = regenerate(MANIFEST_PATH, allow_fixture=ALLOW_FIXTURE)
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
if not ALLOW_FIXTURE and manifest.get('backend') == 'fixture':
    raise ValueError('fixture-only output cannot be reported as a real-model baseline')
summary_rows = aggregate_jsonl(RAW_PATH, expected_scorer='calibrated.v1')
planned_lengths = manifest['context_lengths']
planned_positions = manifest['evidence_positions']
planned_tasks = manifest['task_types']
print({
    'phase': PHASE,
    'backend': manifest['backend'],
    'raw_sha256': regeneration['raw_results_sha256'],
    'summary_rows': len(summary_rows),
    'excluded_cells': regeneration['excluded_cell_n'],
    'baseline_limited': regeneration['baseline_limited_task_types'],
})

In [ ]:
coverage = [
    {key: row[key] for key in (
        'task_type', 'target_context_tokens', 'requested_evidence_position',
        'n', 'completed_n', 'error_n', 'scored_n', 'accuracy',
    )}
    for row in position_curve_rows(summary_rows)
]
coverage

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(planned_tasks), figsize=(15, 4), sharey=True)
axes = [axes] if len(planned_tasks) == 1 else axes
for axis, task_type in zip(axes, planned_tasks):
    for context_tokens in planned_lengths:
        points = [
            row for row in summary_rows
            if row['task_type'] == task_type
            and row['target_context_tokens'] == context_tokens
        ]
        points.sort(key=lambda row: row['requested_evidence_position'])
        axis.plot(
            [row['requested_evidence_position'] for row in points],
            [row['accuracy'] for row in points],
            marker='o',
            label=f'{context_tokens:,} tokens',
        )
    axis.set_title(task_type)
    axis.set_xlabel('requested evidence position')
    axis.set_ylim(0, 1.05)
axes[0].set_ylabel('accuracy')
axes[-1].legend(fontsize='small')
fig.suptitle('Figure A — Position curves')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'position-curves.png', dpi=160)
plt.show()

In [ ]:
gap_rows = position_gap_rows(summary_rows)
gap_rows
fig, axis = plt.subplots(figsize=(8, 4))
for task_type in planned_tasks:
    points = [row for row in gap_rows if row['task_type'] == task_type and row['position_gap'] is not None]
    axis.plot(
        [row['target_context_tokens'] for row in points],
        [row['position_gap'] for row in points],
        marker='o',
        label=task_type,
    )
axis.set_xscale('log', base=2)
axis.set_xlabel('target context tokens')
axis.set_ylabel('A_edge - A_middle')
axis.set_title('Figure B — Position gap')
axis.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'position-gap-vs-context.png', dpi=160)
plt.show()

In [ ]:
fig, axis = plt.subplots(figsize=(8, 4))
for task_type in planned_tasks:
    points = next(
        row['points'] for row in effective_context_by_task(
            [row for row in summary_rows if row['task_type'] == task_type]
        )
        if row['task_type'] == task_type
    )
    axis.plot(
        [point['context_tokens'] for point in points],
        [point['accuracy'] for point in points],
        marker='o',
        label=task_type,
    )
axis.set_xscale('log', base=2)
axis.set_xlabel('target context tokens')
axis.set_ylabel('accuracy')
axis.set_ylim(0, 1.05)
axis.set_title('Figure C — Context degradation')
axis.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'context-degradation.png', dpi=160)
plt.show()

In [ ]:
effective = effective_context_by_task(summary_rows)
[{
    key: row[key] for key in (
        'task_type', 'baseline_accuracy', 'threshold_accuracy', 'status',
        'crossing_context_tokens', 'effective_context_tokens',
        'largest_tested_context_tokens',
    )
} for row in effective]
fig, axis = plt.subplots(figsize=(8, 4))
plotted = [row for row in effective if row['effective_context_tokens'] is not None]
axis.bar(
    [row['task_type'] for row in plotted],
    [row['effective_context_tokens'] for row in plotted],
)
axis.set_ylabel('effective context tokens')
axis.set_title('Figure D — Effective context')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'effective-context-vs-context.png', dpi=160)
plt.show()

In [ ]:
position_sensitive_effective = effective_context_by_task_and_position(summary_rows)
[{
    key: row[key] for key in (
        'task_type', 'evidence_position', 'baseline_accuracy',
        'status', 'crossing_context_tokens', 'effective_context_tokens',
    )
} for row in position_sensitive_effective]

In [ ]:
systems = [
    {
        'task_type': row['task_type'],
        'target_context_tokens': row['target_context_tokens'],
        'median_ttft_s': row['median_ttft_s'],
        'median_prefill_tokens_per_second': row['median_prefill_tokens_per_second'],
        'median_decode_tokens_per_second': row['median_decode_tokens_per_second'],
        'median_peak_memory_bytes': row['median_peak_memory_bytes'],
    }
    for row in summary_rows
]
systems[:5]